# SocraticLoop lightweight post-training pilot

This notebook builds a disjoint 40-train/8-validation synthetic SFT set, performs one QLoRA epoch on Qwen2.5-Coder-3B-Instruct, and evaluates the adapter on the frozen eight-case executable benchmark. Frozen test cases are never used for training.

In [ ]:
REPO_URL = "https://github.com/jasperrrliu-ui/AC215-Project-SocraticLoop.git"
REPO_DIR = "AC215-Project-SocraticLoop"
MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"

In [ ]:
!rm -rf $REPO_DIR
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip -q install -U transformers accelerate bitsandbytes peft datasets trl

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([sys.executable, "scripts/build_post_training_data.py"], check=True)
subprocess.run([sys.executable, "scripts/train_qwen_qlora.py", "--model", MODEL], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/run_qwen_agent_benchmark.py", "--system", "qwen_qlora_harness", "--model", MODEL, "--adapter", "outputs/post_training/qwen_qlora_adapter"], check=True)
subprocess.run([sys.executable, "scripts/score_agent_benchmark.py", "--system", "qwen_qlora_harness", "--responses", "outputs/agent_benchmark/responses/qwen_qlora_harness.jsonl"], check=True)

In [ ]:
from pathlib import Path
import json
report = json.loads(Path("outputs/agent_benchmark/reports/qwen_qlora_harness_report.json").read_text())
print(json.dumps(report["summary"], indent=2))

In [ ]:
!zip -qr post-training-pilot-results.zip outputs/post_training outputs/agent_benchmark/responses/qwen_qlora_harness.jsonl outputs/agent_benchmark/reports/qwen_qlora_harness_report.json
from google.colab import files
files.download("post-training-pilot-results.zip")